### Conda Environment Check

In [1]:
from __future__ import print_function
from packaging.version import parse as Version
from platform import python_version

OK = '\x1b[42m[ OK ]\x1b[0m'
FAIL = "\x1b[41m[FAIL]\x1b[0m"

try:
    import importlib
except ImportError:
    print(FAIL, "Python version 3.12.10 is required,"
                " but %s is installed." % sys.version)

def import_version(pkg, min_ver, fail_msg=""):
    mod = None
    try:
        mod = importlib.import_module(pkg)
        if pkg in {'PIL'}:
            ver = mod.VERSION
        else:
            ver = mod.__version__
        if Version(ver) == Version(min_ver):
            print(OK, "%s version %s is installed."
                  % (lib, min_ver))
        else:
            print(FAIL, "%s version %s is required, but %s installed."
                  % (lib, min_ver, ver))    
    except ImportError:
        print(FAIL, '%s not installed. %s' % (pkg, fail_msg))
    return mod


# first check the python version
pyversion = Version(python_version())

if pyversion >= Version("3.12.10"):
    print(OK, "Python version is %s" % pyversion)
elif pyversion < Version("3.12.10"):
    print(FAIL, "Python version 3.12.10 is required,"
                " but %s is installed." % pyversion)
else:
    print(FAIL, "Unknown Python version: %s" % pyversion)

    
print()
requirements = {'numpy': "2.2.5", 'matplotlib': "3.10.1",'sklearn': "1.6.1", 
                'pandas': "2.2.3",'xgboost': "3.0.0", 'shap': "0.47.2", 
                'polars': "1.27.1", 'seaborn': "0.13.2"}

# now the dependencies
for lib, required_version in list(requirements.items()):
    import_version(lib, required_version)

[ OK ] Python version is 3.12.10

[ OK ] numpy version 2.2.5 is installed.
[ OK ] matplotlib version 3.10.1 is installed.
[ OK ] sklearn version 1.6.1 is installed.
[ OK ] pandas version 2.2.3 is installed.
[ OK ] xgboost version 3.0.0 is installed.
[ OK ] shap version 0.47.2 is installed.
[ OK ] polars version 1.27.1 is installed.
[ OK ] seaborn version 0.13.2 is installed.


### Load the Merged Application Dataset

In [2]:
import pandas as pd
import matplotlib
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_rows', 150)


In [3]:
# read in the dataset
df_full_application = pd.read_csv('../data/application_merged_data.csv')
df_full = df_full_application.copy()
# First view of the dataset
print("Example of the application_merged_data dataset:")
print(df_full.head())
print("\n Shape of the application_merged_data dataset:")
print(df_full.shape)

Example of the application_merged_data dataset:
   SK_ID_CURR  TARGET NAME_CONTRACT_TYPE CODE_GENDER FLAG_OWN_CAR FLAG_OWN_REALTY  CNT_CHILDREN  AMT_INCOME_TOTAL  \
0      100002       1         Cash loans           M            N               Y             0          202500.0   
1      100003       0         Cash loans           F            N               N             0          270000.0   
2      100004       0    Revolving loans           M            Y               Y             0           67500.0   
3      100006       0         Cash loans           F            N               Y             0          135000.0   
4      100007       0         Cash loans           M            N               Y             0          121500.0   

   AMT_CREDIT  AMT_ANNUITY  AMT_GOODS_PRICE NAME_TYPE_SUITE NAME_INCOME_TYPE            NAME_EDUCATION_TYPE  \
0    406597.5      24700.5         351000.0   Unaccompanied          Working  Secondary / secondary special   
1   1293502.5      35698.5 

##### Columns in the Merged Application Dataset

In [15]:
print("Basic Information of the application_merged_data dataset:")
print(df_full.info())
print("\nColumn Names:")
print(df_full.columns.tolist())

Basic Information of the application_merged_data dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Columns: 137 entries, SK_ID_CURR to PREV_STATUS_UNUSED OFFER
dtypes: float64(80), int64(41), object(16)
memory usage: 321.4+ MB
None

Column Names:
['SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'REGION_POPULATION_RELATIVE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'OWN_CAR_AGE', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS', 'REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE

In [18]:
print("\n Description of the application_merged_data dataset:")
print(df_full.describe())


 Description of the application_merged_data dataset:
          SK_ID_CURR         TARGET   CNT_CHILDREN  AMT_INCOME_TOTAL    AMT_CREDIT    AMT_ANNUITY  AMT_GOODS_PRICE  \
count  307511.000000  307511.000000  307511.000000      3.075110e+05  3.075110e+05  307499.000000     3.072330e+05   
mean   278180.518577       0.080729       0.417052      1.687979e+05  5.990260e+05   27108.573909     5.383962e+05   
std    102790.175348       0.272419       0.722121      2.371231e+05  4.024908e+05   14493.737315     3.694465e+05   
min    100002.000000       0.000000       0.000000      2.565000e+04  4.500000e+04    1615.500000     4.050000e+04   
25%    189145.500000       0.000000       0.000000      1.125000e+05  2.700000e+05   16524.000000     2.385000e+05   
50%    278202.000000       0.000000       0.000000      1.471500e+05  5.135310e+05   24903.000000     4.500000e+05   
75%    367142.500000       0.000000       1.000000      2.025000e+05  8.086500e+05   34596.000000     6.795000e+05   
ma

### Missing Values

In [5]:
print("Percentage of Missing:")
missing_percentage = (df_full.isnull().sum() / len(df_full) * 100).sort_values(ascending=False)
print(missing_percentage[missing_percentage > 0])

Percentage of Missing:
COMMONAREA_MODE                 69.872297
COMMONAREA_AVG                  69.872297
COMMONAREA_MEDI                 69.872297
NONLIVINGAPARTMENTS_AVG         69.432963
NONLIVINGAPARTMENTS_MEDI        69.432963
NONLIVINGAPARTMENTS_MODE        69.432963
FONDKAPREMONT_MODE              68.386172
LIVINGAPARTMENTS_MODE           68.354953
LIVINGAPARTMENTS_MEDI           68.354953
LIVINGAPARTMENTS_AVG            68.354953
FLOORSMIN_MODE                  67.848630
FLOORSMIN_AVG                   67.848630
FLOORSMIN_MEDI                  67.848630
YEARS_BUILD_MEDI                66.497784
YEARS_BUILD_MODE                66.497784
YEARS_BUILD_AVG                 66.497784
OWN_CAR_AGE                     65.990810
LANDAREA_AVG                    59.376738
LANDAREA_MEDI                   59.376738
LANDAREA_MODE                   59.376738
BASEMENTAREA_MEDI               58.515956
BASEMENTAREA_MODE               58.515956
BASEMENTAREA_AVG                58.515956
EXT_SOURCE_

In [7]:
# Missing percentage > 50%
high_missing = missing_percentage[missing_percentage > 50]
print(f"Number of columns which contains above 50% missing data: {len(high_missing)} .")
# print("high_missing:",high_missing) 

# 20% < Missing percentage < 50%
medium_missing = missing_percentage[(missing_percentage > 20) & (missing_percentage <= 50)]
print(f"Number of columns which contains 20% to 50% missing data: {len(medium_missing)} .")
# print("medium_missing:",medium_missing) 

# Missing percentage < 20%
low_missing = missing_percentage[(missing_percentage > 0) & (missing_percentage <= 20)]
print(f"Number of columns which contains below 20% missing data: {len(low_missing)} .")
# print("low_missing:",low_missing) 

Number of columns which contains above 50% missing data: 41 .
Number of columns which contains 20% to 50% missing data: 9 .
Number of columns which contains below 20% missing data: 17 .


### Data Preprocessing

In [4]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

##### Continuous, Ordinal, Categorical Features

In [5]:
cols_to_drop = ['TARGET', 'SK_ID_CURR', 'CODE_GENDER']
# Check the cols to drop exist
cols_to_drop = [c for c in cols_to_drop if c in df_full.columns]

X = df_full.drop(columns=cols_to_drop)
y = df_full['TARGET']

print("X Shape:",X.shape)
print("y Shape:",y.shape)

X Shape: (307511, 134)
y Shape: (307511,)


In [6]:
all_features = X.columns.tolist()
all_numerical_features = X.select_dtypes(include=np.number).columns.tolist()
all_categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()
print("Number of all_featuress:", len(all_features))
print("Number of all_numerical_featuress:", len(all_numerical_features))
print("Number of all_categorical_features:", len(all_categorical_features))

Number of all_featuress: 134
Number of all_numerical_featuress: 119
Number of all_categorical_features: 15


In [7]:
binary_features = [col for col in df_full.columns.tolist() if df_full[col].nunique() == 2]
print(f"{len(binary_features)} binary features:", binary_features)

print(f"\n{len(all_categorical_features)} categorical features:", all_categorical_features)
unique_edu = df_full['NAME_EDUCATION_TYPE'].unique()
print("Unique Education Levels:", unique_edu)

37 binary features: ['TARGET', 'NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'FLAG_MOBIL', 'FLAG_EMP_PHONE', 'FLAG_WORK_PHONE', 'FLAG_CONT_MOBILE', 'FLAG_PHONE', 'FLAG_EMAIL', 'REG_REGION_NOT_LIVE_REGION', 'REG_REGION_NOT_WORK_REGION', 'LIVE_REGION_NOT_WORK_REGION', 'REG_CITY_NOT_LIVE_CITY', 'REG_CITY_NOT_WORK_CITY', 'LIVE_CITY_NOT_WORK_CITY', 'EMERGENCYSTATE_MODE', 'FLAG_DOCUMENT_2', 'FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_4', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_7', 'FLAG_DOCUMENT_8', 'FLAG_DOCUMENT_9', 'FLAG_DOCUMENT_10', 'FLAG_DOCUMENT_11', 'FLAG_DOCUMENT_12', 'FLAG_DOCUMENT_13', 'FLAG_DOCUMENT_14', 'FLAG_DOCUMENT_15', 'FLAG_DOCUMENT_16', 'FLAG_DOCUMENT_17', 'FLAG_DOCUMENT_18', 'FLAG_DOCUMENT_19', 'FLAG_DOCUMENT_20', 'FLAG_DOCUMENT_21']

15 categorical features: ['NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCES

In [8]:
suspicious_features = []
for col in [feature for feature in all_numerical_features if feature not in binary_features]:
    nunique = df_full[col].nunique()
    if 2 < nunique <= 10:
        suspicious_features.append(col)
        print(f"Feature name: '{col}'")
        print(f"Count of unique values: {nunique}")
        print(f"Sample of unique values: {np.sort(df_full[col].dropna().unique())[:5]}\n")

Feature name: 'REGION_RATING_CLIENT'
Count of unique values: 3
Sample of unique values: [1 2 3]

Feature name: 'REGION_RATING_CLIENT_W_CITY'
Count of unique values: 3
Sample of unique values: [1 2 3]

Feature name: 'DEF_30_CNT_SOCIAL_CIRCLE'
Count of unique values: 10
Sample of unique values: [0. 1. 2. 3. 4.]

Feature name: 'DEF_60_CNT_SOCIAL_CIRCLE'
Count of unique values: 9
Sample of unique values: [0. 1. 2. 3. 4.]

Feature name: 'AMT_REQ_CREDIT_BUREAU_HOUR'
Count of unique values: 5
Sample of unique values: [0. 1. 2. 3. 4.]

Feature name: 'AMT_REQ_CREDIT_BUREAU_DAY'
Count of unique values: 9
Sample of unique values: [0. 1. 2. 3. 4.]

Feature name: 'AMT_REQ_CREDIT_BUREAU_WEEK'
Count of unique values: 9
Sample of unique values: [0. 1. 2. 3. 4.]

Feature name: 'PREV_CODE_REJECT_REASON_NUNIQUE'
Count of unique values: 8
Sample of unique values: [0. 1. 2. 3. 4.]

Feature name: 'PREV_NAME_CASH_LOAN_PURPOSE_NUNIQUE'
Count of unique values: 9
Sample of unique values: [0. 1. 2. 3. 4.]



In [9]:
ordinal_num_features = ['REGION_RATING_CLIENT', 'REGION_RATING_CLIENT_W_CITY']
ordinal_str_features = ['NAME_EDUCATION_TYPE']

education_order = [
    'Lower secondary', 
    'Secondary / secondary special', 
    'Incomplete higher', 
    'Higher education', 
    'Academic degree'
]

In [44]:
print(df_full[binary_features].head())

   TARGET NAME_CONTRACT_TYPE FLAG_OWN_CAR FLAG_OWN_REALTY  FLAG_MOBIL  FLAG_EMP_PHONE  FLAG_WORK_PHONE  \
0       1         Cash loans            N               Y           1               1                0   
1       0         Cash loans            N               N           1               1                0   
2       0    Revolving loans            Y               Y           1               1                1   
3       0         Cash loans            N               Y           1               1                0   
4       0         Cash loans            N               Y           1               1                0   

   FLAG_CONT_MOBILE  FLAG_PHONE  FLAG_EMAIL  REG_REGION_NOT_LIVE_REGION  REG_REGION_NOT_WORK_REGION  \
0                 1           1           0                           0                           0   
1                 1           1           0                           0                           0   
2                 1           1           0           

In [14]:
df_binary_subset = df_full[binary_features]
binary_features_num = df_binary_subset.select_dtypes(include=np.number).columns.tolist()
binary_features_str = df_binary_subset.select_dtypes(exclude=np.number).columns.tolist()
print("Number of all_binary_features:", len(binary_features))
print("Number of binary_features_num:", len(binary_features_num))
print("Number of binary_features_str:", len(binary_features_str))

cts_features = [f for f in all_numerical_features if f not in ordinal_num_features]
manual_skewed_features = [
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'PREV_AMT_ANNUITY_MEAN', 'PREV_AMT_ANNUITY_SUM', 'PREV_AMT_APPLICATION_MEAN',
    'PREV_AMT_APPLICATION_SUM', 'PREV_AMT_CREDIT_MEAN', 'PREV_AMT_CREDIT_SUM'
]
cts_features_log = [f for f in manual_skewed_features if f in cts_features]
cts_features_other = [f for f in cts_features if f not in cts_features_log]

ord_features_num = ordinal_num_features
ord_features_str = ordinal_str_features
ord_features = ord_features_num + ord_features_str

cat_features = [f for f in all_categorical_features if f not in ord_features]

print(f"Summary:")
print(f"1. Continuous Features :  {len(cts_features)}")
print(f"2. Categorical Features: {len(cat_features)}")
print(f"3. Ordinal Features:     {len(ord_features)}")
print(f"4. Binary Features:      {len(binary_features)}")
print(f"5. Total Counts:         {len(all_features)}")

Number of all_binary_features: 37
Number of binary_features_num: 33
Number of binary_features_str: 4
Summary:
1. Continuous Features :  117
2. Categorical Features: 14
3. Ordinal Features:     3
4. Binary Features:      37
5. Total Counts:         134


In [15]:
import pickle
import pandas as pd


# ==========================================
# 1. Check Variables
# ==========================================

feature_metadata = {
    'all_features': all_features,

    'cts_features': cts_features,
    'cts_features_log': cts_features_log,
    'cts_features_other': cts_features_other,

    'cat_features': cat_features,

    'ord_features_num': ord_features_num,
    'ord_features_str': ord_features_str,
    'ord_features': ord_features,
    'education_order': education_order,
    
    'binary_features': binary_features,
    'binary_features_num': binary_features_num,
    'binary_features_str': binary_features_str,
}

# ==========================================
# 2. Save Feature Lists
# ==========================================
with open('../data/feature_metadata.pkl', 'wb') as f:
    pickle.dump(feature_metadata, f)

print("✅ Feature lists saved to 'feature_metadata.pkl'")
print(f"   - cts_features count: {len(cts_features)}")
print(f"   - cat_features count: {len(cat_features)}")
print(f"   - ord_features count: {len(ord_features)}")


✅ Feature lists saved to 'feature_metadata.pkl'
   - cts_features count: 117
   - cat_features count: 14
   - ord_features count: 3


##### Missing Points in Each Categories

In [47]:
import pandas as pd
import numpy as np


def analyze_missing_values(df, feature_list, feature_type):
    """
    Analysis the missing values of each column in the feature list of the selected feature type 
    """
    print(f"Missing Data Point in the {feature_type} feature :")
    
    subset_df = df[feature_list]
    
    missing_stats = subset_df.isnull().sum()
    missing_stats = missing_stats[missing_stats > 0]
    
    if missing_stats.empty:
        print(f"No missing points in {feature_type} features\n")
        return

    missing_df = pd.DataFrame({
        'Missing Count': missing_stats,
        'Missing Percentage': (missing_stats / len(df) * 100).round(2)
    })
    
    print(missing_df.sort_values(by='Missing Percentage', ascending=False))
    print("\n")


analyze_missing_values(df_full, binary_features, "Binary")
analyze_missing_values(df_full, cat_features, "Categorical")
analyze_missing_values(df_full, ord_features, "Ordinal")
analyze_missing_values(df_full, cts_features, "Continuous")

Missing Data Point in the Binary feature :
                     Missing Count  Missing Percentage
EMERGENCYSTATE_MODE         145755                47.4


Missing Data Point in the Categorical feature :
                     Missing Count  Missing Percentage
FONDKAPREMONT_MODE          210295               68.39
WALLSMATERIAL_MODE          156341               50.84
HOUSETYPE_MODE              154297               50.18
EMERGENCYSTATE_MODE         145755               47.40
OCCUPATION_TYPE              96391               31.35
NAME_TYPE_SUITE               1292                0.42


Missing Data Point in the Ordinal feature :
No missing points in Ordinal features

Missing Data Point in the Continuous feature :
                              Missing Count  Missing Percentage
COMMONAREA_AVG                       214865               69.87
COMMONAREA_MODE                      214865               69.87
COMMONAREA_MEDI                      214865               69.87
NONLIVINGAPARTMENTS_MODE